# Train TrOCR on IAM Handwriting Dataset

This notebook fine-tunes the Microsoft TrOCR model on the IAM Handwriting dataset using Hugging Face datasets (`Teklia/IAM-line`).

### ⚠️ IMPORTANT: Enable GPU ⚠️
Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU NOT Detected! Please change runtime type to GPU.")
    # raise RuntimeError("No GPU found. Training will be too slow.")

In [ ]:
# 1. Clone or Update the repository
import os

if os.path.exists('handwriting_recog'):
    %cd handwriting_recog
    !git pull origin main
else:
    !git clone https://github.com/Bhuvan-018/handwriting_recog
    %cd handwriting_recog

In [ ]:
# 2. Install dependencies
!pip install -r requirements.txt

In [ ]:
# 3. Run the training script (SKIP THIS if you already have the model)
# !python train_hf.py

In [ ]:
# 4. Deploy Existing Model to Hugging Face Space
# Run this cell to deploy your ALREADY TRAINED model from the zip file.

import os
import shutil

# --- Configuration ---
HF_TOKEN = "YOUR_HF_WRITE_TOKEN" # @param {type:"string"}
SPACE_ID = "bhuvan-018/handwriting-recognition" # @param {type:"string"}

# Clean inputs
HF_TOKEN = HF_TOKEN.strip()
SPACE_ID = SPACE_ID.strip()

# Absolute path to the zip file in Colab
ZIP_FILE_PATH = "/content/handwriting_recog/trocr_finetuned_iam_hf.zip"
MODEL_DIR = "models/trocr_finetuned_iam_hf"

# Ensure we are in the correct root directory
if os.path.exists("/content/handwriting_recog"):
    os.chdir("/content/handwriting_recog")
    print(f"📂 Working directory set to: {os.getcwd()}")

# --- 1. Prepare Model ---
if not os.path.exists(MODEL_DIR):
    if os.path.exists(ZIP_FILE_PATH):
        print(f"Found zip file at: {ZIP_FILE_PATH}. Unzipping...")
        !unzip -o {ZIP_FILE_PATH} -d .
        
        # --- CRITICAL: Delete zip to save space ---
        print(f"🗑️ Deleting zip file to free up space: {ZIP_FILE_PATH}")
        os.remove(ZIP_FILE_PATH)
    else:
        # Try relative path just in case
        relative_zip = "trocr_finetuned_iam_hf.zip"
        if os.path.exists(relative_zip):
             print(f"Found zip file at relative path: {relative_zip}. Unzipping...")
             !unzip -o {relative_zip} -d .
             print(f"🗑️ Deleting zip file to free up space: {relative_zip}")
             os.remove(relative_zip)
        else:
             print(f"❌ Error: Zip file not found at {ZIP_FILE_PATH} or current directory.")
             # raise FileNotFoundError("Model zip file not found")

# Verify extraction
if not os.path.exists(MODEL_DIR):
    print(f"❌ Error: Model directory {MODEL_DIR} does not exist after unzipping.")
else:
    # --- 2. Reduce Size: Remove Checkpoints ---
    print("🧹 Cleaning up intermediate checkpoints to save space...")
    checkpoints = [d for d in os.listdir(MODEL_DIR) if d.startswith('checkpoint-')]
    for ckpt in checkpoints:
        ckpt_path = os.path.join(MODEL_DIR, ckpt)
        print(f"Removing {ckpt_path}...")
        shutil.rmtree(ckpt_path)

# --- 3. Deploy to Spaces ---
if HF_TOKEN == "YOUR_HF_WRITE_TOKEN" or not HF_TOKEN:
    print("⚠️ Please enter your Hugging Face Write Token above!")
else:
    print("🚀 Starting deployment to Hugging Face Space...")
    
    # Clean up any existing space_repo to prevent nesting issues
    if os.path.exists("space_repo"):
        print("🗑️ Removing existing space_repo directory...")
        shutil.rmtree("space_repo")
    
    # Install Git LFS
    !git lfs install
    
    # Configure Git
    !git config --global user.email "colab@example.com"
    !git config --global user.name "Colab User"
    
    # Clone your Hugging Face Space
    repo_url = f"https://{HF_TOKEN}@huggingface.co/spaces/{SPACE_ID}"
    !git clone {repo_url} space_repo
    
    # Check if app_gradio.py exists in CURRENT directory (not space_repo)
    if not os.path.exists("app_gradio.py"):
        print("❌ Error: app_gradio.py not found in current directory!")
        print("Current directory:", os.getcwd())
        print("Listing current directory files:")
        print(os.listdir("."))
    else:
        # Copy model files to the Space repo
        print("📦 Copying model files...")
        !mkdir -p space_repo/models/trocr_finetuned_iam_hf
        !cp -r {MODEL_DIR}/* space_repo/models/trocr_finetuned_iam_hf/
        
        # Copy app files (ensure they are up to date from the repo)
        print("📄 Copying app files...")
        !cp app_gradio.py space_repo/app.py
        !cp requirements.txt space_repo/
        !cp -r utils space_repo/
        
        # Commit and Push
        print("⬆️ Pushing to Hugging Face (this may take a few minutes)...")
        # Change directory to space_repo ONLY for git operations
        os.chdir("space_repo")
        !git lfs track "*.bin"
        !git lfs track "*.safetensors"
        !git add .
        !git commit -m "Deploy fine-tuned model from Colab"
        
        # Use robust push
        !git push
        print("✅ Successfully deployed to Hugging Face Space!")
        # Go back to parent directory
        os.chdir("..")